In [ ]:
# ==========================================================
# Smart Cultural Storyteller (SCS) Pipeline
# ==========================================================

# ----------------------------------------------------------
# 0. Requirements (for Google Colab / Local Setup)
# ----------------------------------------------------------
!apt -y install ffmpeg
!apt-get install -y fonts-noto
!wget -O NotoSansDevanagari-Regular.ttf \
    https://github.com/google/fonts/raw/main/ofl/notosansdevanagari/NotoSansDevanagari-Regular.ttf

!pip install -q \
    gradio sentence-transformers faiss-cpu deep-translator \
    transformers bitsandbytes accelerate torch torchvision \
    torchaudio diffusers[torch] safetensors openai-whisper \
    SpeechRecognition gTTS pydub moviepy pysrt wikipedia \
    beautifulsoup4 edge-tts langdetect reportlab

# ----------------------------------------------------------
# 1. Imports & Logging
# ----------------------------------------------------------
import os
import re
import json
import uuid
import logging
import random
import asyncio
from pathlib import Path
from typing import List, Tuple, Optional, Dict

import torch
from pydub import AudioSegment

# Setup logging
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
log = logging.getLogger("SCS")

# ----------------------------------------------------------
# 1.1. Google Drive Setup (Optional)
# ----------------------------------------------------------
try:
    # Mount Google Drive
    from google.colab import drive
    drive.mount("/content/drive")

    # Set project root
    project_root = "/content/drive/MyDrive/SmartCulturalStoryteller"

    # Set Hugging Face cache directory to Google Drive
    os.environ["TRANSFORMERS_CACHE"] = "/content/drive/MyDrive/hf_models"

except:
    pass


# Hugging Face Token (optional)
HF_TOKEN = os.environ.get("HUGGINGFACE_HUB_TOKEN", None)



# ----------------------------------------------------------
# 2. Paths & Configuration
# ----------------------------------------------------------
WORKDIR = Path("./SmartCulturalStoryteller")
KB_DIR = WORKDIR / "knowledgebase"/"datasets"
OUT_DIR = WORKDIR / "outputs"
ASSETS_DIR = WORKDIR / "Assets"

# Create necessary directories
for d in (KB_DIR, OUT_DIR, ASSETS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Configuration
CFG = {
    "embed_model": "sentence-transformers/all-MiniLM-L6-v2",
    "llm_model": "tiiuae/Falcon-H1-0.5b-instruct",
    "sd_model": "stabilityai/stable-diffusion-xl-base-1.0",

    "aspect_map": {
        "16:9": (1280, 720),
        "4:3":  (1024, 768),
        "3:2":  (1200, 800),
        "1:1":  (768, 768),
        "9:16": (720, 1280),
    },
}


# ----------------------------------------------------------
# 3. Language Helpers
# ----------------------------------------------------------
from deep_translator import GoogleTranslator
from langdetect import detect as lang_detect

LANG_CODE_MAP = {"english": "en", "hindi": "hi", "maithili":"mai"}


def lang_to_code(name: str) -> str:
    """Normalize a language name to a short code (default to 'en')."""
    if not name:
        return "en"
    n = name.strip().lower()
    return LANG_CODE_MAP.get(n, n[:2])


def translate_text(text: str, target_lang: str = "en") -> str:
    """Translate using deep_translator with safe fallback to original text."""
    try:
        tgt = target_lang if len(target_lang) <= 3 else lang_to_code(target_lang)
        return GoogleTranslator(source="auto", target=tgt).translate(text)
    except Exception as e:
        return text


def detect_language_simple(text: str) -> str:
    """Attempt several detection methods; return language code."""

    # 1) langdetect
    try:
        return lang_detect(text)

    except Exception:

       if re.search(r"[\u0900-\u097F]", text):
          return "hi"

    return "en"

# ----------------------------------------------------------
# 4. Speech-to-text transcription
# ----------------------------------------------------------
try:
    import whisper
    whisper_model = None

    def load_whisper(model_name: str = "small"):
        global whisper_model
        if whisper_model is None:
            log.info("Loading Whisper model: %s", model_name)
            whisper_model = whisper.load_model(model_name)
        return whisper_model

    def transcribe_audio_whisper(audio_path: str, lang_code: Optional[str] = None) -> str:
        model = load_whisper("small")
        res = model.transcribe(audio_path)
        return res.get("text", "").strip()

except Exception:
    whisper_model = None

    def transcribe_audio_whisper(*args, **kwargs):
        raise RuntimeError("Whisper not available. Install openai-whisper to enable transcription.")

# ----------------------------------------------------------
# 5. Embeddings & Retrieval (FAISS) — file-based -> corpus -> wiki
# ----------------------------------------------------------
try:
    from sentence_transformers import SentenceTransformer
    import faiss
    import numpy as np

    log.info("Loading embedding model: %s", CFG["embed_model"])
    embed_model = SentenceTransformer(CFG["embed_model"]) if CFG.get("embed_model") else None
except Exception as e:
    log.warning("Embeddings not available: %s. RAG disabled.", e)
    embed_model = None
    faiss = None

# Read local knowledge files (if available)
file_docs: List[Dict[str, str]] = []

if embed_model is not None:
    for p in KB_DIR.glob("**/*"):
        if p.suffix.lower() in [".txt", ".md"]:
            try:
                t = p.read_text(encoding="utf-8", errors="ignore").strip()
                if t:
                    # use filename as source
                    source_name = p.stem
                    file_docs.append({"text": t, "source": source_name})
            except Exception as e:
                log.warning("Failed to read %s: %s", p, e)

file_index = None
if embed_model is not None and file_docs:
    try:
        file_emb = embed_model.encode(
            [d["text"] for d in file_docs],
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        d = file_emb.shape[1]
        file_index = faiss.IndexFlatIP(d)
        file_index.add(file_emb)
        log.info("File-based FAISS index ready (%d docs)", len(file_docs))
    except Exception as e:
        log.warning("FAISS build failed: %s", e)
        file_index = None


# fallback  Mini Corpus
MINI_CORPUS = [
    #(Mahabharata, Ramayana, Puranas)
    {
        "title": "Bhishma’s Terrible Vow",
        "text": "To ensure his father Shantanu’s happiness, Devavrata vowed lifelong celibacy and renounced the throne. This sacrifice earned him the name Bhishma and made him one of the most respected figures of the Mahabharata.",
        "source": "Mahabharata",
    },
    {
        "title": "Arjuna Wins Draupadi",
        "text": "At Draupadi’s swayamvara, suitors had to shoot the eye of a rotating fish by looking at its reflection in water. Arjuna, disguised as a Brahmin, succeeded and won Draupadi’s hand in marriage.",
        "source": "Mahabharata",
    },
    {
        "title": "Birth of Rama",
        "text": "King Dasharatha of Ayodhya performed a sacred ritual to have children. The gods blessed him, and Rama, along with his brothers Bharata, Lakshmana, and Shatrughna, were born—marking the beginning of the Ramayana story.",
        "source": "Ramayana",
    },
    {
        "title": "Birth of Krishna",
        "text": "Born in Mathura to Devaki and Vasudeva, Krishna was carried across the Yamuna to Gokul, where he was raised by Nanda and Yashoda.",
        "source": "Bhagavata Purana",
    },
    {
        "title": "Krishna Lifts Govardhan",
        "text": "When Indra sent heavy rains to punish Vrindavan, young Krishna lifted the Govardhan hill on his little finger, providing shelter to the villagers and cattle. This act proved his divinity and protection of devotees.",
        "source": "Bhagavata Purana",
    },
    {
        "title": "Samudra Manthan",
        "text": "Gods and demons churned the ocean of milk to obtain nectar of immortality. Many treasures and dangers emerged, including poison which Shiva consumed. The event highlights cooperation, conflict, and balance in cosmic creation.",
        "source": "Puranas",
    },
    # Gonu Jha
    {
        "title": "Gonu Jha and the Clever Fox",
        "text": "When villagers struggled with a fox stealing chickens, Gonu Jha set a clever trap using feathers and a bucket. The fox was caught and safely relocated, showing his intelligence and problem-solving skills.",
        "source": "Gonu Jha Stories",
    },
    {
        "title": "Gonu Jha and the Trickster Merchant",
        "text": "A merchant tried to cheat villagers with spoiled goods. Gonu Jha pretended to buy everything and then exposed the fraud in public. The merchant fled, and villagers learned to stay cautious.",
        "source": "Gonu Jha Stories",
    },

    # Panchatantra
    {
        "title": "The Monkey and the Crocodile",
        "text": "A crocodile befriended a monkey to steal his heart for his wife. The monkey cleverly saved himself by saying he left his heart on the tree, teaching the value of wit in escaping danger.",
        "source": "Panchatantra",
    },
    {
        "title": "The Lion and the Clever Rabbit",
        "text": "A lion terrorized animals daily, but a clever rabbit tricked him into fighting his own reflection in a well. The lion jumped in and drowned, freeing the forest from his cruelty.",
        "source": "Panchatantra",
    },

    # Akbar-Birbal
    {
        "title": "Birbal’s Khichdi",
        "text": "To prove a man could survive a cold night, Birbal made him stand in water. Later, Birbal cooked khichdi under the sun as a clever metaphor, teaching Akbar patience and fair judgment.",
        "source": "Akbar-Birbal",
    },
    {
        "title": "Counting Crows",
        "text": "Akbar asked Birbal how many crows were in the city. Birbal answered with witty confidence, saying the number matched what Akbar wanted, proving his humor and presence of mind.",
        "source": "Akbar-Birbal",
    },
]


mini_corpus = [f"{s['title']} {s['text']}" for s in MINI_CORPUS]
mini_index = None
if embed_model is not None:
    try:
        mini_emb = embed_model.encode(mini_corpus, convert_to_numpy=True, normalize_embeddings=True)
        d = mini_emb.shape[1]
        mini_index = faiss.IndexFlatIP(d)
        mini_index.add(mini_emb)
    except Exception as e:
        log.warning("Mini corpus embedding failed: %s", e)
        mini_index = None

# Wikipedia helpers
import wikipedia
from bs4 import BeautifulSoup



# ----------------------------------------------------------
# clean text
# ----------------------------------------------------------

def clean_text(text: str) -> str:
    # 1. Remove HTML tags, scripts, and styles
    try:
        soup = BeautifulSoup(text, "html.parser")
        for tag in soup(["script", "style", "sup", "table"]):
            tag.decompose()
        text = soup.get_text(" ")
    except Exception:
        pass

    # 2. Remove markdown formatting (**bold**, __italic__, etc.)
    text = re.sub(r"(\*\*|__)(.*?)\1", r"\2", text)
    text = re.sub(r"(\*|_)(.*?)\1", r"\2", text)

    # 3. Remove AI/system tokens and XML-like tags
    text = re.sub(r"<\s*\/?\s*\w+.*?>", " ", text)
    text = re.sub(r"<\|.*?\|>", " ", text)
    text = re.sub(r"\[/?INST\]", " ", text)
    text = re.sub(r"(\<s\>|\<\/s\>|\<pad\>|\<bos\>|\<eos\>)", " ", text)

    # 4. Remove references like (1), [1], [a], etc.
    text = re.sub(r"\([^)]*\)", "", text)
    text = re.sub(r"\[[^\]]*\]", "", text)

    # 5. Remove redundant symbols & spaces
    text = re.sub(r"[#*_~`]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text





# Wikipedia as fallback
def wiki_fallback(query: str, sentences: int = 3) -> str:
    key = query.lower().replace("tell me","").replace("write a story","")

    try:
        wikipedia.set_lang("en")
        summary = wikipedia.summary(query, sentences=sentences)
        clean = clean_text(summary)
    except Exception:
        try:
            page = wikipedia.page(query, auto_suggest=True)
            clean = clean_text(page.html())
        except Exception as e:
            log.warning("Wikipedia fallback failed for %s: %s", query, e)
            clean = ""

    return clean



def decide_mode_from_context(context: str, source: str = "") -> str:
    PRESERVE_KEYWORDS = [
        "ramayana", "mahabharata", "bhagavad", "krishna", "rama", "arjuna",
        "quran", "bible", "gita", "upanishad", "vedas", "puranas",
        "history", "mythology", "epic", "legend", "ashoka", "gandhi"
    ]

    # 1. Source-based decision (mini corpus or filename if available)
    if source:
        if any(kw in source.lower() for kw in PRESERVE_KEYWORDS):
            return "preserved"
        else:
            return "creative"

    # 2. Context-based decision (file corpus, wiki)
    if context and any(kw in context.lower() for kw in PRESERVE_KEYWORDS):
        return "preserved"

    # 3. Default
    return "creative"

def retrieve_context(query: str, top_k: int = 3, sim_threshold: float = 0.60) -> Tuple[str, str]:
    """
    Try file-based RAG -> mini-corpus -> Wikipedia fallback. Returns (context, mode).
    """
    query_emb = embed_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)

    # 1) File-based RAG
    if file_index is not None and file_docs:
        try:
            D, I = file_index.search(query_emb, k=top_k)
            if D[0][0] >= sim_threshold:
                idx = I[0][0]
                context = file_docs[idx]["text"]
                source = file_docs[idx].get("source", "")
                mode = decide_mode_from_context(context, source)
                return context, mode
        except Exception as e:
            log.warning("File index search failed: %s", e)

    # 2) Mini-corpus
    if mini_index is not None and MINI_CORPUS:
        try:
            D, I = mini_index.search(query_emb, k=top_k)
            if D[0][0] >= sim_threshold:
                idx = I[0][0]
                context = f"{MINI_CORPUS[idx]['title']} {MINI_CORPUS[idx]['text']}"
                source = MINI_CORPUS[idx].get("source", "")
                mode = decide_mode_from_context(context, source)
                return context, mode
        except Exception as e:
            log.warning("Mini index search failed: %s", e)

    # 3) Wiki fallback
    wiki = wiki_fallback(query, sentences=4)
    if wiki:
        mode = decide_mode_from_context(wiki)
        return wiki, mode

    return "", "creative"


# ----------------------------------------------------------
# 6. Story LLM wrapper (text generation)
# ----------------------------------------------------------
from transformers import AutoTokenizer, AutoModelForCausalLM
model=None
tokenizer=None
try:
  log.info("Attemp to load LLM...")
  tokenizer = AutoTokenizer.from_pretrained(CFG.get("llm_model"))
  model = AutoModelForCausalLM.from_pretrained(CFG.get("llm_model"))

except Exception as e:
  log.warning("Language model/tokenizer loading failed!")
  model,tokenizer=None,None

def generate_text(prompt: str, max_new_tokens: int = 400, temperature: float = 0.6) -> str:
    """Generate text """
    if model is None or tokenizer is None:
        log.warning("LLM model/tokenizer not initialized !")
        return " "

    try:
      inputs = tokenizer.apply_chat_template(
      prompt,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt", ).to(model.device)

      outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      temperature=temperature,
      top_p=0.95,
      do_sample=True,
      )
      story = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:])

      return story


    except Exception as e:
        log.error("default generator failed: %s", e)
        return " "




# build prompt template
def structured_prompt( user_prompt: str, num_words: int = 300, context: str = "", mode: str = "Preserved") -> list:

    # --- Mode Instructions ---
    if mode.lower().startswith("pr"):
        mode_instruct = (
            "You must faithfully retell traditional stories. "
            "Keep narration clear, simple, and engaging. "
            "If historical, mythological, or sacred context is provided, preserve accuracy. "
            "Do not invent or alter established facts. " )
    else:
        mode_instruct = (
            "You are free to co-create engaging stories inspired by the user's idea. "
            "Be imaginative but remain respectful of cultural values. ")


    # --- System Message ---
    system_msg = {
        "role": "system",
        "content": (
            "## You are 'Smart Cultural Storyteller'.\n"
            "### Core Instructions:\n"
            f"- {mode_instruct}\n"
            "- Tell the story in smooth narrative form.\n"
            "- Use natural flow: beginning → challenge → resolution → lesson.\n"
            "- Do not label story parts (like 'introduction' or 'conflict').\n"
            "- Do not explain the process. Only output the story text.\n"
            "- Keep story length around the requested word count."
        )
    }

    # --- User Message ---
    user_msg = {
        "role": "user",
        "content": (
            f"User Idea: {user_prompt}\n\n"
            f"Context (for grounding, if any): {context}\n\n"
            f"Story length: **Keep story length under {num_words} words.**"
        )
    }

    return [system_msg, user_msg]




# Story generator
def generate_story(user_prompt: str, input_audio_path:str="", story_len: str = "small", story_lang:str="English") -> str:


    if not user_prompt:
          raise ValueError("No prompt text provided.")

    # translate user prompt to English for LLM
    user_prompt_en = translate_text(user_prompt, target_lang="en")

    # Context Retrieval
    context, mode = retrieve_context(user_prompt_en)

    # Get number of words and tokens
    story_len_norm = (story_len or "small").lower()
    if story_len_norm in ["large", "long"]:
        max_length = 1200
        num_words = 1000
    elif story_len_norm == "medium":
        max_length = 700
        num_words = 500
    else:
        max_length = 400
        num_words = 300

    # structured user prompt
    prompt = structured_prompt(user_prompt_en, num_words , context, mode)

    # generate story
    try:
        yield "✍️ Writing story..."
        out = generate_text(prompt, max_new_tokens=max_length, temperature=0.8)
        story_text = clean_text(out)
        yield story_text
        #translate story into requested lagnguage
        target_code = lang_to_code(story_lang)
        story_localized = translate_text(story_text, target_lang=target_code)
        yield story_localized.strip()
    except Exception as e:
        log.error("Story generation failed: %s", e)
        yield f"Opps! Error{e}\n\n Story generation failed, Try again..."






# ----------------------------------------------------------
# 7. Scene chunking
# ----------------------------------------------------------
def split_into_scenes(story: str, per_scene_dur: int = 5, wpm: int = 150) -> List[str]:
    """Split story into scene-sized chunks. Conservative algorithm using sentence boundaries."""
    if not story:
        return []

    sentences = re.split(r"(?<=[.!?।])\s+", story.strip())
    sentences = [s.strip() for s in sentences if s.strip()]

    MAX_SENTENCES = 2
    chunks = []
    i = 0
    while i < len(sentences):
        chunk = sentences[i : i + MAX_SENTENCES]
        chunks.append(" ".join(chunk))
        i += MAX_SENTENCES

    if not chunks:
        return [story.strip()]
    return chunks

# image prompt builder
def image_prompt_from_scene(story_chunks: List[str], style: str) -> List[str]:
    STYLE_TAGS = {
    "storybook": "storybook illustration, whimsical, pastel tones, fairy-tale aesthetic, vibrant colors",
    "mythological": "Raja Ravi Varma style, Indian traditional attire, epic scene, ornate jewelry, divine aura",
    "painting": "oil painting, rich texture, brush strokes, renaissance shading, chiaroscuro lighting",
    "cartoon": "cartoon style, clean outlines, vibrant colors, smooth shading, Pixar Disney style",
    "anime": "anime style, vibrant colors, clean line art, dynamic poses, cinematic lighting",
    "3d animation": "3d render, ultra-detailed, cinematic lighting, Unreal Engine style, realistic shaders",
    "photo realistic": "photorealistic, ultra-detailed, hyperreal textures, 8k resolution, cinematic lighting"
    }

    tag = STYLE_TAGS.get(style.lower(), "")
    refined_img_prompts: List[str] = []
    try:

      for chunk in story_chunks:

          # translate cunk to English
          chunk = translate_text(chunk, target_lang="en")

          prompt_text =prompt = [{
          "role": "system",
          "content": (
              "#You are a **visual prompt generator** for image creation.\n"
              "Your role: extract vivid visual description **visual elements** from the given story text.\n"
              "## Instructions:\n"
              "- Identify characters with details appearance visual description.\n"
              "- Identify the background/setting.\n"
              "- Capture the main action or emotion.\n"
              "- Use rich, descriptive adjectives.\n"
              "- Do NOT retell the story or add extra details.\n"
              "- Output should be one vivid visual description **under 20 words**only."
              f"here is story text: '{chunk}'\n"
              "provide image Prompt here:"
          )
          }]

          img_prompt = generate_text(prompt_text, max_new_tokens=30, temperature=0.7)
          img_prompt = clean_text(img_prompt).strip()
          if not img_prompt:
                raise ValueError("Empty prompt returned")
          refined_img_prompts.append(img_prompt)
    except Exception as e:
          log.warning("image prompt generation failed for a scene: %s", e)
          refined_img_prompts.append(chunk[:120])
    final_img_prompts = [f"{prompt}, {tag}, highly detailed, sharp , vivid colors" for prompt in refined_img_prompts]
    print(final_img_prompts)
    return final_img_prompts


# ----------------------------------------------------------
# 8. Image generation
# ----------------------------------------------------------
try:
    from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
    from PIL import ImageDraw, ImageFont, Image as PILImage
    SD_PIPE = None
    try:
        log.info("Attempting to load Stable Diffusion model")
        sd = StableDiffusionXLPipeline.from_pretrained(CFG.get("sd_model"),
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            use_auth_token=HF_TOKEN,
        )
        sd.scheduler = DPMSolverMultistepScheduler.from_config(sd.scheduler.config)
        sd = sd.to("cuda" if torch.cuda.is_available() else "cpu")
        SD_PIPE = sd
        log.info("Stable Diffusion model loaded.")
    except Exception as e:
        log.warning("Stable Diffusion model load failed: %s", e)
        SD_PIPE = None
except Exception:
    SD_PIPE = None



def generate_image(prompt: str, size: Tuple[int, int], seed: Optional[int] = None):
    """Generate an image via SD if available, else return a placeholder image with the prompt text."""


    if SD_PIPE:
        try:
            device = "cuda" if torch.cuda.is_available() else "cpu"
            generator = torch.Generator(device=device).manual_seed(seed) if seed is not None else None
            negative = "duplicate, watermark, text, blurry, low quality, bad anatomy,deformed face, disfigured, deformed, mutated, extra limbs, extra fingers, poorly drawn,nsfw, nuidity, gore"
            img = SD_PIPE(prompt, negative_prompt=negative, height=size[1], width=size[0], num_inference_steps=40, guidance_scale=7.5, generator=generator).images[0]
            return img
        except Exception as e:
            log.warning("Stable Diffusion generation failed: %s", e)

    # Placeholder image
    img = PILImage.new("RGB", size, color=(255, 255, 240))
    draw = ImageDraw.Draw(img)
    short = (prompt[:200] + "...") if len(prompt) > 200 else prompt
    try:
        draw.text((20, 20), short, fill=(0, 0, 0))
    except Exception:
        pass
    return img

# ----------------------------------------------------------
# 9. text to speech generation
# ----------------------------------------------------------
import asyncio
import nest_asyncio
from gtts import gTTS
from pydub import AudioSegment

nest_asyncio.apply()  # Allow nested event loops in notebooks/Gradio

try:
    import edge_tts
    EDGE_AVAILABLE = True
except Exception:
    EDGE_AVAILABLE = False


async def edge_tts_save(text: str, voice: str, out_path: str):
    communicate = edge_tts.Communicate(text, voice=voice)
    await communicate.save(out_path)


def run_async_blocking(coro):
    """Run async coroutine safely in notebook/Gradio environment."""
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(coro)


def tts_generate(text: str, gender: str, out_path: str):
    voice_map = {
        ("en", "male"): "en-IN-ArjunNeural",
        ("en", "female"): "en-IN-NeerjaNeural",
        ("hi", "male"): "hi-IN-MadhurNeural",
        ("hi", "female"): "hi-IN-SwaraNeural",
    }
    # 1) Language detection
    detected_script_code = detect_language_simple(text)
    log.info("Detected language: %s", detected_script_code )

    lang_code = detected_script_code

    lang2 = (lang_code or "en")[:2].lower()
    v = voice_map.get((lang2, gender.lower()), None)

    # Try Edge-TTS first
    if EDGE_AVAILABLE and v:
        try:
            run_async_blocking(edge_tts_save(text, v, out_path))
            return out_path
        except Exception as e:
            print(f"⚠️ Edge-TTS failed: {e}")

    # Fallback to gTTS
    try:
        tld = "co.in" if lang2 == "en" else "com"
        g = gTTS(text=text, lang=lang2, tld=tld)
        g.save(out_path)
        return out_path
    except Exception as e:
        print(f"⚠️ gTTS failed: {e}")
        silent = AudioSegment.silent(duration=1000)
        silent.export(out_path, format="mp3")
        return out_path



# ----------------------------------------------------------
# 10. Captions (SRT) builder
# ----------------------------------------------------------
try:
    import pysrt
except Exception:
    pysrt = None


def ms_to_srt_time(ms: int):
    h = ms // (3600 * 1000)
    ms_rem = ms % (3600 * 1000)
    m = ms_rem // (60 * 1000)
    ms_rem = ms_rem % (60 * 1000)
    s = ms_rem // 1000
    ms_final = ms_rem % 1000
    return pysrt.SubRipTime(hours=h, minutes=m, seconds=s, milliseconds=ms_final)


def build_srt(story_text: str, audio_path: Path, srt_out: Path):
    if pysrt is None:
        raise RuntimeError("pysrt not installed; cannot build SRT.")

    # Split story into sentences (supports Devanagari punctuation too)
    sentences = re.split(r"(?<=[.!?।])\s+", story_text)
    sentences = [s.strip() for s in sentences if s.strip()]

    audio = AudioSegment.from_file(str(audio_path))
    total_ms = len(audio)
    total_words = sum(len(s.split()) for s in sentences) or 1

    subs = pysrt.SubRipFile()
    current_ms = 0

    for i, s in enumerate(sentences, 1):
        word_count = len(s.split())
        dur_ms = int(total_ms * (word_count / total_words))
        start_ms = current_ms
        end_ms = min(current_ms + dur_ms, total_ms - 1)
        subs.append(
            pysrt.SubRipItem(
                index=i,
                start=ms_to_srt_time(start_ms),
                end=ms_to_srt_time(end_ms),
                text=s,
            )
        )
        current_ms += dur_ms

    subs.save(str(srt_out), encoding="utf-8")
    return srt_out

# ----------------------------------------------------------
# 11. Final Video assembly
# ----------------------------------------------------------
try:
    import requests
    import subprocess
    from moviepy.editor import (
        ImageClip,
        concatenate_audioclips,
        concatenate_videoclips,
        vfx,
        AudioFileClip,
        CompositeAudioClip,
    )
except Exception as e:
    log.warning("moviepy not available: %s", e)
    ImageClip = None
    AudioFileClip = None
    concatenate_videoclips = None
    CompositeAudioClip = None




# royalty-free background music URLs
bg_music_urls = {
    # Flute & Spiritual
    "spiritual_krishna_flute_peaceful_divine": "https://archive.org/download/krishnaflu-pa2luqdy-37180/krishnaflu-pa2luqdy-37180.mp3",
    "meditation_bapu_flute_calm_serenity": "https://ia801604.us.archive.org/1/items/BapuFluteMeditationWithBansuri/BapuFlute-Meditate2.mp3",
    # Ambient & Fantasy
    "fantasy_ambient_magical_wonderland": "https://archive.org/download/ambientforfilm/Fantasy.flac",
    "harmony_ambient_peaceful_emotional": "https://archive.org/download/ambientforfilm/Harmony.flac",
    # Lo-Fi & Chill
    "lofi_calm_chill_relaxing_peaceful": "https://archive.org/download/royalty-free-music/Cozy%20Strings%20(Lofi%20Chill).mp3",
    "lofi_relax_focus_study_background": "https://ia801403.us.archive.org/6/items/royalty-free-music/Relax%20Chill%20Hop%20%28Lo-Fi%20Chill%29.mp3",
    # Nature & Meditation
    "nature_ambient_calm_forest_rain": "https://ia801403.us.archive.org/6/items/royalty-free-music/Fading%20Memories%20%28Ambient%29.mp3",
    # Action & Adventure
    "battle_action_chase_danger_thrill": "https://dn721600.ca.archive.org/0/items/royalty-free-music/Battle.mp3",
    "cinematic_fast_future_sci_fi_thrill": "https://ia601403.us.archive.org/6/items/royalty-free-music/Classic%20Award%20%28Cinematic%20Background%20Music%29.mp3",
    "epic_heroic_inspiration_victory_power": "https://dn721600.ca.archive.org/0/items/royalty-free-music/Epic%20Inspiration.mp3",
    # Kids & Storytelling
    "kids_funny_cartoon_happy_fantasy": "https://dn721600.ca.archive.org/0/items/royalty-free-music/Ukulele%20Bliss%20%28Kids%20Game%29.mp3",
}

bg_music_path = os.path.join(ASSETS_DIR, "background_musics")
os.makedirs(bg_music_path, exist_ok=True)

# Download music if not exists
for name, url in bg_music_urls.items():
    file_path = os.path.join(bg_music_path, f"{name}.mp3")
    if not os.path.exists(file_path):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            with open(file_path, "wb") as f:
                f.write(r.content)
        except Exception as e:
              pass

def pick_best_music(scene_chunk, bg_music_path):
    """
    Pick the best matching music file for a given story chunk.
    Uses semantic similarity on filenames.
    """
    # List all music files
    music_files = [f for f in os.listdir(bg_music_path) if f.endswith(".mp3")]
    if not music_files:
        return None

    # Encode the story chunk
    scene_emb = embed_model.encode([scene_chunk])[0]

    # Encode music filenames (without extension)
    music_names = [os.path.splitext(f)[0] for f in music_files]
    music_embs = embed_model.encode(music_names)

    # Compute cosine similarity
    sims = np.dot(music_embs, scene_emb) / (
        np.linalg.norm(music_embs, axis=1) * np.linalg.norm(scene_emb)
    )

    # Pick top music
    best_idx = int(np.argmax(sims))
    return os.path.join(bg_music_path, music_files[best_idx])


from typing import List, Optional
from pathlib import Path
import os, random, requests, subprocess
import cv2
from moviepy.editor import (
    ImageClip, AudioFileClip, concatenate_videoclips,
    concatenate_audioclips, CompositeAudioClip
)
import math


def assemble_video(
    image_paths: List[Path],
    audio_paths: List[Path],
    out_video: Path,
    fps: int = 24,
    captions: Optional[Path] = None,
    burn: bool = False,
    background_music: bool = False,
    bg_music: str = "",
    aspect_ratio:str='16:9',
    font_path: str = "/usr/share/fonts/truetype/noto/NotoSansDevanagari-Regular.ttf"
):
    if ImageClip is None:
        raise RuntimeError("moviepy/ffmpeg is required to assemble video.")

    clips, narration_tracks = [], []
    n = min(len(image_paths), len(audio_paths))
    if n == 0:
        raise ValueError("No images or audio to assemble.")

    target_w, target_h = CFG["aspect_map"].get(aspect_ratio, (1280, 720))
    print(target_w,target_h)

    for idx, (img_p, aud_p) in enumerate(zip(image_paths[:n], audio_paths[:n])):
        audio_clip = AudioFileClip(str(aud_p))
        narration_tracks.append(audio_clip)

        base_clip = ImageClip(str(img_p)).resize(height=target_h * 1)
        clip = base_clip.set_duration(audio_clip.duration)

        # Random effect
        zoom_type = random.choice(["in", "out", "none"])
        pan_type = random.choice(["lr", "rl", "tb", "bt", "diag1", "diag2", "none"])
        zoom_factor = random.uniform(1.1, 1.2)

        if zoom_type == "in":
            start_scale, end_scale = 1.0, zoom_factor
        elif zoom_type == "out":
            start_scale, end_scale = zoom_factor, 1.0
        else:
            start_scale = end_scale = 1.0

        pan_px = 80
        if pan_type == "lr":
            dx, dy = pan_px, 0
        elif pan_type == "rl":
            dx, dy = -pan_px, 0
        elif pan_type == "tb":
            dx, dy = 0, pan_px
        elif pan_type == "bt":
            dx, dy = 0, -pan_px
        elif pan_type == "diag1":
            dx, dy = pan_px, pan_px
        elif pan_type == "diag2":
            dx, dy = -pan_px, -pan_px
        else:
            dx, dy = 0, 0

        def crop_func(get_frame, t):
            progress = t / clip.duration
            progress = 0.5 * (1 - math.cos(math.pi * progress))

            scale = start_scale + (end_scale - start_scale) * progress
            frame = get_frame(t)
            H, W = frame.shape[:2]

            w = int(target_w / scale)
            h = int(target_h / scale)

            cx, cy = W // 2, H // 2
            x = cx - w // 2 + int(dx * progress)
            y = cy - h // 2 + int(dy * progress)

            x = max(0, min(x, W - w))
            y = max(0, min(y, H - h))

            crop = frame[y:y+h, x:x+w]
            return cv2.resize(crop, (target_w, target_h))

        clip = clip.fl(crop_func, apply_to=["mask"]).set_audio(audio_clip)

        if idx > 0:
            clip = clip.crossfadein(1.5)

        clips.append(clip)

    final = concatenate_videoclips(clips, method="compose")
    narration_audio = concatenate_audioclips(narration_tracks)

    bg_music_audio = None
    if bg_music:
        bg_music_audio = AudioFileClip(bg_music)
        if bg_music_audio.duration > final.duration:
            bg_music_audio = bg_music_audio.subclip(0, final.duration)
        else:
            loops = int(final.duration // bg_music_audio.duration) + 1
            bg_music_audio = concatenate_audioclips([bg_music_audio] * loops).subclip(0, final.duration)

        bg_music_audio = bg_music_audio.audio_fadein(1).audio_fadeout(1).volumex(0.1)

    final_audio = CompositeAudioClip([bg_music_audio, narration_audio]) if bg_music_audio else narration_audio
    final = final.set_audio(final_audio)

    # audiobook as MP3
    audiobook_path = str(out_video).replace(".mp4", "_audiobook.mp3")
    final_audio.write_audiofile(audiobook_path, fps=44100, codec="mp3")

    temp_video = str(out_video).replace(".mp4", "_temp.mp4")
    final.write_videofile(temp_video, fps=fps, codec="libx264", audio_codec="aac")
    final.close()

    # Caption handling
    if captions and os.path.exists(str(captions)):
        try:
            # Convert .srt → .ass (better font shaping for Devanagari)
            ass_file = str(captions).replace(".srt", ".ass")
            subprocess.run(["ffmpeg", "-y", "-i", str(captions), ass_file], check=True)

            if burn:
                # Burn with libass + font
                cmd = [
                    "ffmpeg", "-y", "-i", temp_video,
                    "-vf", f"ass={ass_file}:fontsdir={os.path.dirname(font_path)}",
                    "-c:a", "copy", str(out_video)
                ]
            else:
                # Soft embed (mov_text works for Roman only, not for Devanagari)
                cmd = [
                    "ffmpeg", "-y", "-i", temp_video,
                    "-vf", f"ass={ass_file}:fontsdir={os.path.dirname(font_path)}",
                    "-c:v", "libx264", "-c:a", "aac",
                    str(out_video)
                ]

            subprocess.run(cmd, check=True)
            os.remove(temp_video)
        except Exception as e:
            print(f"Caption embedding failed: {e}")
    else:
        os.replace(temp_video, str(out_video))

    return out_video, audiobook_path



from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch

# ----------------------------------------------------------
# StoryBook builder
# ----------------------------------------------------------
def save_storybook_pdf(paragraphs, image_paths, filename="storybook.pdf"):
    """Creates a storybook-style PDF with paragraphs and images."""
    try:
        # Create PDF document
        doc = SimpleDocTemplate(
            filename,
            pagesize=letter,
            rightMargin=50,
            leftMargin=50,
            topMargin=50,
            bottomMargin=50
        )

        # Define paragraph styles
        styles = getSampleStyleSheet()
        paragraph_style = ParagraphStyle(
            "StoryParagraph",
            parent=styles["Normal"],
            fontName="Helvetica",
            fontSize=12,
            leading=18,
            spaceAfter=12,
            alignment=4  # Justify text
        )

        elements = []

        # Add paragraphs and images
        for i, para in enumerate(paragraphs):
            # Add paragraph text
            elements.append(Paragraph(para, paragraph_style))
            elements.append(Spacer(1, 0.2 * inch))

            # Add image if available
            if i < len(image_paths) and os.path.exists(str(image_paths[i])):
                try:
                    # Open image to calculate correct aspect ratio
                    with PILImage.open(image_paths[i]) as img:
                        img_width, img_height = img.size
                        max_width, max_height = 5 * inch, 3.5 * inch

                        # Maintain aspect ratio
                        ratio = min(max_width / img_width, max_height / img_height)
                        new_width = img_width * ratio
                        new_height = img_height * ratio

                    rl_img = RLImage(str(image_paths[i]), width=new_width, height=new_height)
                    rl_img.hAlign = "CENTER"
                    elements.append(rl_img)
                    elements.append(Spacer(1, 0.3 * inch))

                except Exception as e:
                    print(f"⚠️ Image error for {image_paths[i]}: {e}")

        # Build the PDF
        doc.build(elements)
        return filename

    except Exception as e:
        print("Error generating PDF:", e)
        return None





# ----------------------------------------------------------
# 12. Orchestrator: End-to-end pipeline
# ----------------------------------------------------------
from PIL import Image, ImageDraw


def run_pipeline(
    prompt_text: str,
    input_audio_path: Optional[str],
    story_text:str = "",
    story_language:str = "English",
    story_length: str = "Small",
    narration_gender: str = "Female",
    narration_language: str = "--select--",
    aspect_ratio: str = "16:9",
    captions: bool = False,
    scene_style: str = "Mythological",
    background_music: bool = False,
) -> Dict:

    session_id = uuid.uuid4().hex[:8]
    session_dir = OUT_DIR / f"run_{session_id}"
    session_dir.mkdir(parents=True, exist_ok=True)
    log.info("New session: %s", session_dir)

    try:
      if not story_text:
        if prompt_text :
            try:
              yield "Genarating Story...", " Writing story...",None, None , None, None
              # Generate story
              story_text = generate_story(prompt_text, story_len=story_length, story_lang=story_language)
            except:
              yield "⚠️Story generation failed or returned empty text. try again...", None, None, None , None, None

      # if story text given
      if story_text:
        yield "Spliting story into Scenes...", story_text, None, None , None, None
        # 2) Split story text into scenes chunk and prepare image prompts
        scenes_chunks = split_into_scenes(story_text)

      if not scenes_chunks and story_text:
        scenes_chunks = [story_text]


      yield "creating scenes visuals...", story_text, None, None , None, None

      image_prompts = image_prompt_from_scene(scenes_chunks, style=scene_style)

      # 3) Generate images (or placeholders) and save
      resolution = CFG["aspect_map"].get(aspect_ratio, (1280, 720))
      print(resolution)
      image_paths: List[Path] = []
      for i, prompt in enumerate(image_prompts, start=1):
          try:
              img = generate_image(prompt, resolution)
          except Exception as e:
              log.warning("Image generation failed for scene %d: %s", i, e)
              img = Image.new("RGB", resolution, color=(255, 255, 240))
              draw = ImageDraw.Draw(img)
              draw.text((20, 20), prompt[:200], fill=(0, 0, 0))

          ip = session_dir / f"scene_{i:02d}.png"
          img.save(ip)
          image_paths.append(ip)

      # creating storybook
      storybook_file=save_storybook_pdf(scenes_chunks, image_paths)


      bg_music=None
      # pick background music if enable
      if background_music:
        yield "selecting background music...", story_text, None, None , None, None
        try:
          bg_music = pick_best_music(image_prompts[0], bg_music_path)
        except:
          pass


      # Localize scenes to requested language for audio narration (optional)
      if narration_language != "--select--":
        target_code = lang_to_code(narration_language)
        scenes_localized = [translate_text(s, target_lang=target_code) for s in scenes_chunks]
      else:
        scenes_localized = scenes_chunks


      yield "creating scenes visuals...", story_text, None, None , None, None
      # 8) Generate per-scene narration audio (optional) AND single full narration
      audio_paths: List[Path] = []
      for i, txt in enumerate(scenes_localized, start=1):
          ap = session_dir / f"scene_{i:02d}.mp3"
          try:
              tts_generate(txt, gender=narration_gender, out_path=str(ap))
          except Exception as e:
              log.warning("TTS failed for scene %d: %s", i, e)
              silent = AudioSegment.silent(duration=1000)
              silent.export(str(ap), format="mp3")
          audio_paths.append(ap)


      # generate a single text/audio narration file for preview/export
      full_story_text = " ".join(scenes_localized)
      full_audio_narration = session_dir / "story_audiobook.mp3"

      # 9) Build captions SRT (proportional timing) if requested
      srt_path = None
      if captions:
          yield "Preparing Captions...", story_text, None, None , None, None
          if pysrt is None:
              log.warning("pysrt not installed; skipping SRT generation.")
          else:
              srt_path = session_dir / "captions.srt"
              try:
                  tts_generate(full_story_text,  gender=narration_gender, out_path=str(full_audio_narration))
              except Exception as e:
                  log.warning("Full-story TTS failed: %s", e)
                  silent = AudioSegment.silent(duration=1000)
                  silent.export(str(full_audio_narration), format="mp3")

              build_srt(full_story_text, full_audio_narration, srt_path)
      else:
          try:
              tts_generate(full_story_text, gender=narration_gender, out_path=str(full_audio_narration))
          except Exception as e:
              log.warning("Full-story TTS failed: %s", e)
              silent = AudioSegment.silent(duration=1000)
              silent.export(str(full_audio_narration), format="mp3")


      # 10) Assemble video (use per-scene audio tracks in the visual timing)
      out_video = session_dir / "story_video.mp4"
      try:
          yield "Assembling Multimedia...", story_text, None, None , None, None
          final_video,audiobook=assemble_video(
              image_paths=image_paths,
              audio_paths=audio_paths,
              out_video=out_video,
              captions=srt_path if captions else None,
              burn=captions,
              bg_music=bg_music,
              aspect_ratio=aspect_ratio
          )
      except Exception as e:
          log.error("Video assembly failed: %s", e)
          out_video = None

      result = {
          "session_dir": str(session_dir),
          "story": full_story_text,
          "storybook":storybook_file,
          "audio": audiobook,
          "srt": str(srt_path) if srt_path else None,
          "video": str(out_video) if out_video else None,
      }

      log.info("Session complete: %s", result["video"])
      yield "✅Story Video Successfully Generated!", result['story'],result['storybook'],result['srt'],result['audio'],result['video']

    except Exception as e:
          log.exception("Storyteller Pipeline failed")
          yield f"⚠️FError: {e}", result['story'],result['storybook'],result['srt'],result['audio'],result['video']

# ----------------------------------------------------------
# 13. Gradio UI
# ----------------------------------------------------------
try:
    import gradio as gr
except Exception:
    gr = None


if gr is not None:
    def gradio_ui():
        with gr.Blocks(title="Smart Cultural Storyteller") as demo:
            gr.Markdown("# Smart Cultural Storyteller")

            region = gr.Dropdown(choices=["India", "USA", "China"], value="India", label="Region:")


            with gr.Row():
                with gr.Column(scale=1):
                    audio_in = gr.Audio(sources=["microphone"], type="filepath", label="🎤 Voice Prompt", scale=1)
                    txt_in = gr.Textbox(label="Prompt (story idea)", placeholder="Write your idea here...", lines=2)
                    story_lang = gr.Dropdown(choices=["English", "Hindi", "Maithili"], value="English", label="Story Language")
                    length = gr.Radio(["Small", "Medium", "Long"], value="Medium", label="Story Length")
                    gen_story_btn = gr.Button("Generate Story ❇️")
                    story_text = gr.Textbox(label="Story Text", placeholder="Generated story will appear here. Or paste your own.",lines=20, max_lines=20)


                with gr.Column(scale=2):
                    with gr.Row():
                        voice = gr.Radio(["Male", "Female"], value="Female", label="Narration voice")
                        narr_lang = gr.Dropdown(choices=["--select--","English", "Hindi", "Maithili"], value="--select--", label="Narration Language(optional)")
                        status = gr.Textbox(label="status",value="Ready!",interactive=False)

                    with gr.Row():
                        captions_checkbox = gr.Checkbox(label="Add video captions", value=False)
                        background_music = gr.Checkbox(label="background music", value=False)

                    with gr.Row():
                        aspect_ratio = gr.Radio(["16:9", "4:3", "3:2","1:1","9:16"], value="4:3", label="aspect ratio")
                        style = gr.Dropdown(["Storybook", "Cartoon", "Anime", "Painting", "Photo Realistic","3d Animation","Mythological"], value="Storybook", label="Art style")


                    video_gen_btn = gr.Button("Generate Story Video")
                    video_out = gr.Video(label="Final Video",width=1024,height=576)


            # 📂 Media Preview Section
            gr.Markdown("## 📂 Media:")

            with gr.Row():
              storybook = gr.File(label="Download Storybook (PDF)")
              srt_file = gr.File(label="Download Captions (SRT)")
              audiobook = gr.File(label="Download Audiobook (MP3)")


            # Automatically transcribe after recording stops
            audio_in.stop_recording(
                fn=transcribe_audio_whisper,
                inputs=audio_in,
                outputs=txt_in
            )



            # Generate story
            gen_story_btn.click(
                fn=generate_story,
                inputs=[txt_in, length, story_lang],
                outputs=story_text
            )


            # Generate story video
            video_gen_btn.click(
               fn=run_pipeline,
                inputs=[txt_in, audio_in, story_text, length,story_lang, voice, narr_lang, captions_checkbox, style, aspect_ratio, background_music],
                outputs=[status, story_text, storybook, srt_file, audiobook, video_out],
            )

            demo.launch()



if __name__ == '__main__':
    gradio_ui()


